In [0]:
# Databricks notebook source

# COMMAND ----------
# MAGIC %md
# MAGIC # 🥈 SILVER — Limpeza e Enriquecimento dos Dados F1
# MAGIC
# MAGIC **Origem:** `bronze_resultados_f1`
# MAGIC **Destino:** `silver_resultados_f1`
# MAGIC
# MAGIC Transformações aplicadas:
# MAGIC - Tipos de dados corretos (datas, inteiros, floats)
# MAGIC - Flags derivadas: venceu, pódio, pole, terminou
# MAGIC - Limpeza de nulos
# MAGIC - Padronização de strings

# COMMAND ----------

from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, FloatType

DATABASE      = "f1_db"
TABELA_BRONZE = "bronze_resultados_f1"
TABELA_SILVER = "silver_resultados_f1"

spark.sql(f"USE {DATABASE}")

# COMMAND ----------

# Lê da Bronze
df = spark.table(f"{DATABASE}.{TABELA_BRONZE}")
print(f"📥 Registros na Bronze: {df.count()}")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Transformações de tipo e limpeza

# COMMAND ----------

df_silver = (
    df
    # ── Tipos corretos ──────────────────────────────────────────────────────
    .withColumn("ano",          F.col("ano").cast(IntegerType()))
    .withColumn("rodada",       F.col("rodada").cast(IntegerType()))
    .withColumn("data",         F.to_date(F.col("data"), "yyyy-MM-dd"))
    .withColumn("pontos",       F.col("pontos").cast(FloatType()))
    .withColumn("posicao",      F.col("posicao").cast(IntegerType()))
    .withColumn("grid",         F.col("grid").cast(IntegerType()))
    .withColumn("numero_carro", F.col("numero_carro").cast(IntegerType()))
    .withColumn("voltas",       F.col("voltas").cast(IntegerType()))

    # ── Limpeza de strings ──────────────────────────────────────────────────
    .withColumn("piloto",         F.trim(F.col("piloto")))
    .withColumn("construtor",     F.trim(F.col("construtor")))
    .withColumn("codigo_piloto",  F.upper(F.trim(F.col("codigo_piloto"))))
    .withColumn("status",         F.trim(F.col("status")))
    .fillna({"codigo_piloto": "N/A", "status": "Unknown", "grid": 0})

    # ── Flags derivadas ─────────────────────────────────────────────────────
    .withColumn("terminou",      F.col("status") == "Finished")
    .withColumn("pontuou",       F.col("pontos") > 0)
    .withColumn("pole_position", F.col("grid") == 1)
    .withColumn("venceu",        F.col("posicao") == 1)
    .withColumn("podio",         F.col("posicao") <= 3)
    .withColumn("pontos_sem_sprint",
        F.when(F.col("tipo") == "corrida", F.col("pontos")).otherwise(F.lit(0.0))
    )

    # ── Semestre/mês para analytics ─────────────────────────────────────────
    .withColumn("mes",           F.month(F.col("data")))
    .withColumn("semestre",
        F.when(F.month(F.col("data")) <= 6, "1S").otherwise("2S")
    )

    # ── Metadados Silver ────────────────────────────────────────────────────
    .withColumn("_silver_ts", F.current_timestamp())

    # ── Remove metadados da Bronze ──────────────────────────────────────────
    .drop("_source_file", "_ingestion_ts", "_ingestion_date")
)

# COMMAND ----------

display(df_silver.limit(10))
print(f"📤 Registros na Silver: {df_silver.count()}")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Validação de qualidade

# COMMAND ----------

# Checar nulos em colunas críticas
from pyspark.sql.functions import col, count, when

colunas_criticas = ["ano", "rodada", "corrida", "piloto", "construtor", "posicao", "pontos"]
checks = df_silver.select([
    count(when(col(c).isNull(), c)).alias(f"nulls_{c}")
    for c in colunas_criticas
])
print("🔍 Nulos em colunas críticas:")
display(checks)

# COMMAND ----------

# Grava Silver como Delta Table particionada por ano
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("ano")
    .saveAsTable(f"{DATABASE}.{TABELA_SILVER}")
)

print(f"✅ Tabela '{DATABASE}.{TABELA_SILVER}' criada com partição por 'ano'!")

# COMMAND ----------

# Resumo final
spark.sql(f"""
    SELECT ano, tipo, COUNT(*) AS registros,
           COUNT(DISTINCT piloto) AS pilotos,
           COUNT(DISTINCT corrida) AS corridas
    FROM {DATABASE}.{TABELA_SILVER}
    GROUP BY ano, tipo
    ORDER BY ano, tipo
""").show()

📥 Registros na Bronze: 819


ano,tipo,rodada,corrida,circuito,pais,data,piloto,codigo_piloto,construtor,numero_carro,grid,posicao,pontos,voltas,status,terminou,pontuou,pole_position,venceu,podio,pontos_sem_sprint,mes,semestre,_silver_ts
2025,corrida,1,Australian Grand Prix,Albert Park Grand Prix Circuit,Australia,2025-03-16,Lando Norris,NOR,McLaren,4,1,1,25.0,57,Finished,true,true,true,true,true,25.0,3,1S,2026-06-22T02:48:41.449Z
2025,corrida,1,Australian Grand Prix,Albert Park Grand Prix Circuit,Australia,2025-03-16,Max Verstappen,VER,Red Bull,1,3,2,18.0,57,Finished,true,true,false,false,true,18.0,3,1S,2026-06-22T02:48:41.449Z
2025,corrida,1,Australian Grand Prix,Albert Park Grand Prix Circuit,Australia,2025-03-16,George Russell,RUS,Mercedes,63,4,3,15.0,57,Finished,true,true,false,false,true,15.0,3,1S,2026-06-22T02:48:41.449Z
2025,corrida,1,Australian Grand Prix,Albert Park Grand Prix Circuit,Australia,2025-03-16,Andrea Kimi Antonelli,ANT,Mercedes,12,16,4,12.0,57,Finished,true,true,false,false,false,12.0,3,1S,2026-06-22T02:48:41.449Z
2025,corrida,1,Australian Grand Prix,Albert Park Grand Prix Circuit,Australia,2025-03-16,Alexander Albon,ALB,Williams,23,6,5,10.0,57,Finished,true,true,false,false,false,10.0,3,1S,2026-06-22T02:48:41.449Z
2025,corrida,1,Australian Grand Prix,Albert Park Grand Prix Circuit,Australia,2025-03-16,Lance Stroll,STR,Aston Martin,18,13,6,8.0,57,Finished,true,true,false,false,false,8.0,3,1S,2026-06-22T02:48:41.449Z
2025,corrida,1,Australian Grand Prix,Albert Park Grand Prix Circuit,Australia,2025-03-16,Nico Hülkenberg,HUL,Sauber,27,17,7,6.0,57,Finished,true,true,false,false,false,6.0,3,1S,2026-06-22T02:48:41.449Z
2025,corrida,1,Australian Grand Prix,Albert Park Grand Prix Circuit,Australia,2025-03-16,Charles Leclerc,LEC,Ferrari,16,7,8,4.0,57,Finished,true,true,false,false,false,4.0,3,1S,2026-06-22T02:48:41.449Z
2025,corrida,1,Australian Grand Prix,Albert Park Grand Prix Circuit,Australia,2025-03-16,Oscar Piastri,PIA,McLaren,81,2,9,2.0,57,Finished,true,true,false,false,false,2.0,3,1S,2026-06-22T02:48:41.449Z
2025,corrida,1,Australian Grand Prix,Albert Park Grand Prix Circuit,Australia,2025-03-16,Lewis Hamilton,HAM,Ferrari,44,8,10,1.0,57,Finished,true,true,false,false,false,1.0,3,1S,2026-06-22T02:48:41.449Z


📤 Registros na Silver: 819
🔍 Nulos em colunas críticas:


nulls_ano,nulls_rodada,nulls_corrida,nulls_piloto,nulls_construtor,nulls_posicao,nulls_pontos
0,0,0,0,0,0,0


✅ Tabela 'f1_db.silver_resultados_f1' criada com partição por 'ano'!
+----+-------+---------+-------+--------+
| ano|   tipo|registros|pilotos|corridas|
+----+-------+---------+-------+--------+
|2025|corrida|      479|     21|      24|
|2025| sprint|      120|     21|       6|
|2026|corrida|      154|     22|       7|
|2026| sprint|       66|     22|       3|
+----+-------+---------+-------+--------+

